# 02_preparation

Objectif : produire des jeux d'entraînement, de validation et de test propres, sans fuite de données.

Décisions appliquées (voir notes/decisions_equipe.md) :
- suppression des groupes à étiquettes contradictoires, puis des doublons ;
- colonnes retirées : id, attack_cat, ct_ftp_cmd ;
- lignes du test identiques à une ligne de l'entraînement retirées ;
- regroupement des protocoles rares calculé sur l'entraînement uniquement.

Noms de fichiers inversés dans cette version du dataset : voir README.

In [ ]:
##chargement des deux partitions :
import pandas as pd
import numpy as np

SEED = 42
BASE = '/kaggle/input/datasets/mrwellsdavid/unsw-nb15/'
TRAIN_FILE = 'UNSW_NB15_testing-set.csv'   # 175 341 lignes : entraînement officiel
TEST_FILE = 'UNSW_NB15_training-set.csv'   # 82 332 lignes : test officiel

train_raw = pd.read_csv(BASE + TRAIN_FILE)
test_raw = pd.read_csv(BASE + TEST_FILE)
print(train_raw.shape, test_raw.shape)

In [ ]:
##Nettoyage : contradictions puis doublons
TARGET = 'label'
DROP_COLS = ['id', 'attack_cat', 'ct_ftp_cmd']
FEATS = [c for c in train_raw.columns if c not in DROP_COLS + [TARGET]]
print(len(FEATS), 'caractéristiques')

def clean(df, name):
    n0 = len(df)
    n_labels = df.groupby(FEATS)[TARGET].transform('nunique')
    df = df[n_labels == 1]
    n1 = len(df)
    df = df.drop_duplicates(subset=FEATS, keep='first')
    n2 = len(df)
    print(f'{name} : départ {n0} | sans contradictions {n1} | sans doublons {n2}')
    return df.reset_index(drop=True)

train = clean(train_raw, 'entraînement')
test = clean(test_raw, 'test')

In [ ]:
##retirer du test les lignes deja presente dans l'entrainement 
m = test.merge(train[FEATS].drop_duplicates(), on=FEATS, how='left', indicator=True)
assert len(m) == len(test)
overlap = (m['_merge'] == 'both').to_numpy()

print("lignes du test identiques à une ligne de l'entraînement :", overlap.sum(), 'sur', len(test))
print("label des lignes retirées :")
print(test.loc[overlap, TARGET].value_counts())

test = test[~overlap].reset_index(drop=True)

print()
print("entraînement :", train.shape, "| part d'attaques :", round(train[TARGET].mean(), 3))
print("test         :", test.shape, "| part d'attaques :", round(test[TARGET].mean(), 3))

In [ ]:
## 4. Découpage entraînement / validation (80 % / 20 %, stratifié sur attack_cat)
from sklearn.model_selection import train_test_split

VAL_SIZE = 0.2

train_fit, val = train_test_split(
    train, test_size=VAL_SIZE, stratify=train['attack_cat'], random_state=SEED
)
train_fit = train_fit.reset_index(drop=True)
val = val.reset_index(drop=True)

for name, d in [('entraînement', train_fit), ('validation', val), ('test', test)]:
    print(f"{name:13s}: {d.shape[0]:6d} lignes | part d'attaques : {d[TARGET].mean():.3f}")

print()
print(pd.concat([
    train_fit['attack_cat'].value_counts().rename('entraînement'),
    val['attack_cat'].value_counts().rename('validation'),
    test['attack_cat'].value_counts().rename('test'),
], axis=1))

# contrôle : aucune ligne de validation identique à une ligne d'entraînement
k_fit = set(pd.util.hash_pandas_object(train_fit[FEATS], index=False))
k_val = pd.util.hash_pandas_object(val[FEATS], index=False)
print()
print("lignes de validation identiques à une ligne d'entraînement :", int(k_val.isin(k_fit).sum()))

In [ ]:
## 5. Regroupement des modalités rares (au moins 100 lignes)
MIN_COUNT = 100
CAT_COLS = ['proto', 'service', 'state']

KEEP = {}
for c in CAT_COLS:
    vc = train_fit[c].value_counts()
    KEEP[c] = vc[vc >= MIN_COUNT].index.tolist()
    rares = [v for v in vc.index if v not in KEEP[c]]
    print(f'{c} : conservées {KEEP[c]}')
    print(f'      regroupées en « autres » : {rares}')

def group_rare(df):
    df = df.copy()
    for c in CAT_COLS:
        df[c] = df[c].where(df[c].isin(KEEP[c]), 'autres')
    return df

train_fit = group_rare(train_fit)
val = group_rare(val)
test = group_rare(test)

for c in CAT_COLS:
    print()
    print(train_fit.groupby(c)[TARGET].agg(['count', 'mean']).rename(
        columns={'count': 'lignes', 'mean': 'taux_attaque'}
    ).sort_values('lignes', ascending=False))

In [ ]:
## 6. Encodage one-hot de proto, service, state
def encode(df):
    df = df.drop(columns=['id', 'ct_ftp_cmd'])
    return pd.get_dummies(df, columns=CAT_COLS, dtype=int)

train_enc = encode(train_fit)
val_raw_enc = encode(val)
test_raw_enc = encode(test)

val_enc = val_raw_enc.reindex(columns=train_enc.columns, fill_value=0)
test_enc = test_raw_enc.reindex(columns=train_enc.columns, fill_value=0)

print('colonnes absentes de l\'entraînement (validation) :', set(val_raw_enc.columns) - set(train_enc.columns))
print('colonnes absentes de l\'entraînement (test)       :', set(test_raw_enc.columns) - set(train_enc.columns))

X_COLS = [c for c in train_enc.columns if c not in (TARGET, 'attack_cat')]
print()
print('nombre de caractéristiques :', len(X_COLS))
print(train_enc.shape, val_enc.shape, test_enc.shape)
print('valeurs manquantes :', train_enc.isna().sum().sum(), val_enc.isna().sum().sum(), test_enc.isna().sum().sum())
print('colonnes non numériques parmi les caractéristiques :', train_enc[X_COLS].select_dtypes(exclude='number').columns.tolist())

In [ ]:
## 7. Sauvegarde et empreintes
import hashlib
import os

OUT = '/kaggle/working/'

def fingerprint(df):
    h = pd.util.hash_pandas_object(df, index=False).values
    return hashlib.md5(h.tobytes()).hexdigest()[:12]

assert train_enc.columns.tolist() == val_enc.columns.tolist() == test_enc.columns.tolist()

sets = [('train', train_enc), ('val', val_enc), ('test', test_enc)]

for name, d in sets:
    d.to_csv(OUT + f'{name}_prepared.csv', index=False)

summary = pd.DataFrame([
    {'ensemble': name, 'lignes': len(d), 'part_attaques': round(d[TARGET].mean(), 3), 'empreinte': fingerprint(d)}
    for name, d in sets
])
summary.to_csv(OUT + 'prep_summary.csv', index=False)

print(summary)
print(os.listdir(OUT))

In [1]:
## 8. Contrôle de reproductibilité : contenu de src/data.py
"""Préparation des données UNSW-NB15 (projet HyperIDS).

Reproduit le notebook 02_preparation. Usage :

    train, val, test, x_cols = load_prepared()
    verify(train, val, test)

Sur Kaggle, copier le contenu de ce fichier dans la première cellule de code
du notebook (les notebooks n'importent pas src/ directement).
Les noms des fichiers du dataset sont inversés dans la version Kaggle utilisée.
"""
import hashlib

import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
BASE = '/kaggle/input/datasets/mrwellsdavid/unsw-nb15/'
TRAIN_FILE = 'UNSW_NB15_testing-set.csv'   # 175 341 lignes : entraînement officiel
TEST_FILE = 'UNSW_NB15_training-set.csv'   # 82 332 lignes : test officiel

TARGET = 'label'
DROP_COLS = ['id', 'attack_cat', 'ct_ftp_cmd']
CAT_COLS = ['proto', 'service', 'state']
MIN_COUNT = 100
VAL_SIZE = 0.2

EXPECTED = {'train': 'fb8d73b1cdba', 'val': 'f5db41df66c1', 'test': '0f96e3a31bcc'}


def _clean(df, feats):
    """Retire les groupes à étiquettes contradictoires, puis les doublons."""
    n_labels = df.groupby(feats)[TARGET].transform('nunique')
    df = df[n_labels == 1]
    df = df.drop_duplicates(subset=feats, keep='first')
    return df.reset_index(drop=True)


def _encode(df):
    df = df.drop(columns=['id', 'ct_ftp_cmd'])
    return pd.get_dummies(df, columns=CAT_COLS, dtype=int)


def load_prepared(base=BASE):
    """Retourne (train, val, test, x_cols). attack_cat reste dans les tableaux
    mais ne fait pas partie des caractéristiques (x_cols)."""
    train_raw = pd.read_csv(base + TRAIN_FILE)
    test_raw = pd.read_csv(base + TEST_FILE)
    feats = [c for c in train_raw.columns if c not in DROP_COLS + [TARGET]]

    train = _clean(train_raw, feats)
    test = _clean(test_raw, feats)

    # retire du test les lignes déjà présentes dans l'entraînement
    m = test.merge(train[feats].drop_duplicates(), on=feats, how='left', indicator=True)
    assert len(m) == len(test)
    overlap = (m['_merge'] == 'both').to_numpy()
    test = test[~overlap].reset_index(drop=True)

    # découpage entraînement / validation, stratifié sur attack_cat
    train_fit, val = train_test_split(
        train, test_size=VAL_SIZE, stratify=train['attack_cat'], random_state=SEED
    )
    train_fit = train_fit.reset_index(drop=True)
    val = val.reset_index(drop=True)

    # modalités conservées : au moins MIN_COUNT lignes dans train_fit
    keep = {}
    for c in CAT_COLS:
        vc = train_fit[c].value_counts()
        keep[c] = vc[vc >= MIN_COUNT].index.tolist()

    def group_rare(df):
        df = df.copy()
        for c in CAT_COLS:
            df[c] = df[c].where(df[c].isin(keep[c]), 'autres')
        return df

    train_fit, val, test = group_rare(train_fit), group_rare(val), group_rare(test)

    train_enc = _encode(train_fit)
    val_enc = _encode(val).reindex(columns=train_enc.columns, fill_value=0)
    test_enc = _encode(test).reindex(columns=train_enc.columns, fill_value=0)

    x_cols = [c for c in train_enc.columns if c not in (TARGET, 'attack_cat')]
    return train_enc, val_enc, test_enc, x_cols


def fingerprint(df):
    h = pd.util.hash_pandas_object(df, index=False).values
    return hashlib.md5(h.tobytes()).hexdigest()[:12]


def verify(train, val, test):
    """Compare les empreintes à celles du notebook 02_preparation."""
    got = {'train': fingerprint(train), 'val': fingerprint(val), 'test': fingerprint(test)}
    for k in got:
        statut = 'OK' if got[k] == EXPECTED[k] else 'DIFFÉRENTE'
        print(f'{k:5s} attendue {EXPECTED[k]} | obtenue {got[k]} | {statut}')
    return got == EXPECTED

In [2]:
## 9. Contrôle de reproductibilité : exécution et vérification
train_c, val_c, test_c, x_cols_c = load_prepared()
print(train_c.shape, val_c.shape, test_c.shape, len(x_cols_c))
print(verify(train_c, val_c, test_c))

(80648, 60) (20163, 60) (52738, 60) 58
train attendue fb8d73b1cdba | obtenue fb8d73b1cdba | OK
val   attendue f5db41df66c1 | obtenue f5db41df66c1 | OK
test  attendue 0f96e3a31bcc | obtenue 0f96e3a31bcc | OK
True
